# D326 — Publish Olist Spark Results to HDFS, Catalog, and MySQL

Choose completed outputs from D323–D325 and publish them. This notebook provides connection and write patterns only; it does not provide any analytical solution.


## Lab rules

- Use Apache Spark **3.5.9**, `spark.sql(...)`, DataFrames, and the Spark session catalog only.
- Source tables are in `olist_silver`; do not read CSV again in analytical notebooks.
- Use CTEs and windows only when the exercise permits or requires them.
- State the grain of every intermediate and final result.
- Prevent multiplication when joining two one-to-many datasets such as items and payments.
- Round displayed currency to two decimals, but do not round intermediate calculations.
- Every answer cell is intentionally empty. This notebook contains no solution SQL.
- Catalog metadata persists in local embedded Derby; use the same `SPARK_CATALOG_DIR` in D322–D326.
- Only one active Spark driver may use this catalog. Stop the prior notebook's Spark session first.


In [ ]:
import os
import socket

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, StringType, StructField, StructType

master_url = os.environ.get(
    "SPARK_MASTER",
    f"spark://{socket.gethostname()}:7077",
)

spark = (
    SparkSession.builder
    .appName("D32-Spark-SQL-Window-Functions")
    .master(master_url)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)
print("Master:", spark.sparkContext.master)


## 1. Select results

Publish at least three results: one basic analytical result, one CTE result, and one window result. Re-run your own SQL or read your saved result tables. Add `loaded_at` and document the output grain.


In [ ]:
# TODO: Build or load your three result DataFrames.
# Do not paste credentials into this notebook.


## 2. HDFS Parquet result

Write at least one result to `hdfs:///user/<user>/olist/results/<result_name>` in Parquet. Choose partition columns only when the result is sufficiently large and commonly filtered by them; prefer year/month over daily partitions. Verify the files and read them back.


In [ ]:
result_root = os.environ.get("OLIST_RESULT_HDFS", f"hdfs:///user/{os.environ['USER']}/olist/results")
# TODO: result_df.write.mode(...).parquet(...)
# TODO: read back and compare schema/count.


## 3. Managed Spark catalog result

Use `saveAsTable("olist_results.<table_name>")`. The persistent Spark session catalog stores metadata in the configured local Derby directory, while managed Parquet data is stored in HDFS (normally `/user/hive/warehouse/olist_results.db/...`, or `/user/spark/warehouse/...` when configured before session creation). Inspect `DESCRIBE EXTENDED` and the HDFS path.


In [ ]:
spark.sql("CREATE DATABASE IF NOT EXISTS olist_results")
# TODO pattern:
# result_df.write.format("parquet").mode("overwrite").saveAsTable("olist_results.<table_name>")
# TODO: validate count, schema, provider, and location.


## 4. MySQL prerequisites

Install one compatible MySQL Connector/J 8 JAR in `$SPARK_HOME/jars`, restart Spark, and set `MYSQL_USERNAME` and `MYSQL_PASSWORD` before starting Jupyter. Optional variables: `MYSQL_HOST`, `MYSQL_PORT`, `MYSQL_DATABASE`. Never print the password.


In [ ]:
required = [k for k in ("MYSQL_USERNAME", "MYSQL_PASSWORD") if not os.environ.get(k)]
if required:
    raise RuntimeError("Set before starting Jupyter: " + ", ".join(required))

mysql_host = os.environ.get("MYSQL_HOST", "localhost")
mysql_port = os.environ.get("MYSQL_PORT", "3306")
mysql_database = os.environ.get("MYSQL_DATABASE", "olist_analytics")
jdbc_url = f"jdbc:mysql://{mysql_host}:{mysql_port}/{mysql_database}?useSSL=false&allowPublicKeyRetrieval=true&serverTimezone=UTC"
jdbc_options = {"user": os.environ["MYSQL_USERNAME"], "password": os.environ["MYSQL_PASSWORD"], "driver": "com.mysql.cj.jdbc.Driver"}
print(f"Target database: mysql://{mysql_host}:{mysql_port}/{mysql_database}")


## 5. Write every selected result to MySQL

Use a separate, clearly named MySQL table for each result. Reduce JDBC writer partitions for these small aggregates. Choose `append` or `overwrite` deliberately, use a batch size, and explain rerun behavior. Do not publish raw customer-level identifiers unless required by the exercise.


In [ ]:
# Pattern only:
# (result_df.coalesce(2).write.format("jdbc")
#  .option("url", jdbc_url).option("dbtable", "<table_name>")
#  .options(**jdbc_options).option("batchsize", "1000")
#  .mode("overwrite").save())

# TODO: Publish all three selected results.


## 6. Read-back validation

Read each MySQL table through JDBC. Assert source/target row counts and compare column sets, null counts, and aggregate control totals. Display a deterministic sample. Also verify the HDFS and catalog copies.


In [ ]:
# TODO: JDBC read-back and assertions for every published result.


## Submission checklist

- At least one explicit HDFS Parquet result.
- At least one managed `saveAsTable` result with catalog and HDFS location evidence.
- Three MySQL result tables (analytical, CTE, window) with read-back validation.
- No credentials committed and no analytical solution copied from D17 resolved notebooks.
